# Extended Data Preparation Analysis Project
## Based on 'Exploratory Data Analysis with Python Cookbook'
### Chapter 2: Preparing Data for EDA

In [ ]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print(f"Pandas version: {pd.__version__}")
print(f"Project initialized successfully!")

---
## PART 1: DATA LOADING AND INSPECTION

In [ ]:
# Configure file paths
DATA_DIR = Path("data")
MAIN_FILE = DATA_DIR / "marketing_campaign.csv"

# Load dataset
def load_marketing_data(filepath):
    """Load and return the marketing campaign dataset"""
    df = pd.read_csv(filepath)
    return df

# Load the data
if MAIN_FILE.exists():
    marketing_data = load_marketing_data(MAIN_FILE)
    print(f"✓ Data loaded successfully! Shape: {marketing_data.shape}")
else:
    print(f"⚠ Warning: Data file not found at {MAIN_FILE}")
    print("Please ensure marketing_campaign.csv exists in the data/ folder")
    marketing_data = None

In [ ]:
# Initial inspection
if marketing_data is not None:
    print("="*70)
    print("FIRST 5 ROWS (Transposed for visibility)")
    print("="*70)
    print(marketing_data.head(5).T)
    
    print("\n" + "="*70)
    print("DATASET INFORMATION")
    print("="*70)
    print(f"Total Rows: {marketing_data.shape[0]}")
    print(f"Total Columns: {marketing_data.shape[1]}")
    print(f"Column Names: {list(marketing_data.columns)}")

In [ ]:
# Data types inspection
if marketing_data is not None:
    print("\n" + "="*70)
    print("COLUMN DATA TYPES")
    print("="*70)
    display(marketing_data.dtypes.to_frame(name='DataType'))
    
    print("\n" + "="*70)
    print("MISSING VALUES CHECK")
    print("="*70)
    missing = marketing_data.isnull().sum()
    missing_pct = (missing / len(marketing_data)) * 100
    missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
    display(missing_df[missing_df['Missing Count'] > 0])

---
## PART 2: DATA PREPARATION TECHNIQUES

In [ ]:
# Subset relevant columns for analysis
if marketing_data is not None:
    selected_columns = ['ID', 'Year_Birth', 'Education', 'Marital_Status', 
                        'Income', 'Kidhome', 'Teenhome', 'Dt_Customer',
                        'Recency', 'NumStorePurchases', 'NumWebVisitsMonth']
    
    # Filter columns that exist
    available_cols = [col for col in selected_columns if col in marketing_data.columns]
    marketing_data_subset = marketing_data[available_cols].copy()
    
    print(f"Subset created with {len(available_cols)} columns")
    print(f"Shape: {marketing_data_subset.shape}")

### Technique 1: GROUPING DATA

In [ ]:
if marketing_data_subset is not None:
    print("\n" + "="*70)
    print("GROUPING: Average Store Purchases by Number of Kids")
    print("="*70)
    
    # Group by Kidhome and calculate mean store purchases
    group_result = marketing_data_subset.groupby('Kidhome')['NumStorePurchases'].agg([
        ('Mean', 'mean'),
        ('Median', 'median'),
        ('Min', 'min'),
        ('Max', 'max'),
        ('Count', 'count')
    ]).round(2)
    
    display(group_result)
    
    # Multi-column grouping
    print("\n" + "="*70)
    print("MULTI-COLUMN GROUPING: By Education & Marital Status")
    print("="*70)
    
    multi_group = marketing_data_subset.groupby(['Education', 'Marital_Status']).agg({
        'Income': 'mean',
        'NumStorePurchases': 'mean'
    }).round(2).sort_values('Income', ascending=False)
    
    display(multi_group.head(10))

### Technique 2: APPENDING DATA (Row-wise Combination)

In [ ]:
# Demonstrate appending with samples from existing data
if marketing_data_subset is not None:
    print("\n" + "="*70)
    print("APPENDING: Combining Data Samples Row-Wise")
    print("="*70)
    
    # Create two sample subsets
    sample1 = marketing_data_subset.iloc[:500].copy()
    sample2 = marketing_data_subset.iloc[500:1000].copy()
    
    print(f"Sample 1 shape: {sample1.shape}")
    print(f"Sample 2 shape: {sample2.shape}")
    
    # Append datasets
    appended_data = pd.concat([sample1, sample2], axis=0)
    print(f"Appended data shape: {appended_data.shape}")
    print(f"Expected: 1000 rows, {len(sample1.columns)} columns ✓")

### Technique 3: CONCATENATING DATA (Column-wise Combination)

In [ ]:
if marketing_data_subset is not None:
    print("\n" + "="*70)
    print("CONCATENATING: Combining Features Column-Wise")
    print("="*70)
    
    # Split features for demonstration
    base_features = marketing_data_subset[['ID', 'Year_Birth', 'Education', 'Income']].copy()
    purchase_features = marketing_data_subset[['ID', 'NumStorePurchases', 'NumWebVisitsMonth']].copy()
    
    print(f"Base features shape: {base_features.shape}")
    print(f"Purchase features shape: {purchase_features.shape}")
    
    # Set ID as index for proper concatenation
    base_features.set_index('ID', inplace=True)
    purchase_features.set_index('ID', inplace=True)
    
    # Concatenate along columns
    concatenated_data = pd.concat([base_features, purchase_features], axis=1)
    print(f"Concatenated data shape: {concatenated_data.shape}")
    display(concatenated_data.head())

### Technique 4: MERGING DATA

In [ ]:
if marketing_data_subset is not None:
    print("\n" + "="*70)
    print("MERGING: Joining Tables on Common Keys")
    print("="*70)
    
    # Create demo tables for merge demonstration
    table1 = marketing_data_subset[['ID', 'Year_Birth', 'Education']].drop_duplicates().head(100).copy()
    table2 = marketing_data_subset[['ID', 'Marital_Status', 'Income']].drop_duplicates().head(100).copy()
    
    print(f"Table 1 (Demographics): {table1.shape}")
    print(f"Table 2 (Financial): {table2.shape}")
    
    # Perform inner merge
    merged_inner = pd.merge(table1, table2, on='ID', how='inner')
    print(f"Inner merge result: {merged_inner.shape}")
    display(merged_inner.head())
    
    # Perform left merge
    merged_left = pd.merge(table1, table2, on='ID', how='left')
    print(f"Left merge result: {merged_left.shape}")

### Technique 5: SORTING DATA

In [ ]:
if marketing_data_subset is not None:
    print("\n" + "="*70)
    print("SORTING: Arranging Data by Key Metrics")
    print("="*70)
    
    # Sort by income descending
    sorted_income = marketing_data_subset.sort_values('Income', ascending=False)
    print("Top 10 Highest Income Customers:")
    display(sorted_income[['ID', 'Income', 'Education', 'NumStorePurchases']].head(10))
    
    # Sort by multiple columns
    sorted_multi = marketing_data_subset.sort_values(
        ['Income', 'NumStorePurchases'], 
        ascending=[False, True]
    )
    print("\nSorted by Income (desc), then Store Purchases (asc):")
    display(sorted_multi[['ID', 'Income', 'NumStorePurchases']].head(10))

### Technique 6: CATEGORIZING DATA (BINNING)

In [ ]:
if marketing_data_subset is not None:
    print("\n" + "="*70)
    print("CATEGORIZING: Creating Bins from Numerical Values")
    print("="*70)
    
    # Bin store purchases
    marketing_data_subset['Purchases_Category'] = pd.cut(
        marketing_data_subset['NumStorePurchases'],
        bins=[0, 4, 8, 13],
        labels=['Low', 'Moderate', 'High']
    )
    
    # Age binning example
    current_year = 2026
    marketing_data_subset['Age'] = current_year - marketing_data_subset['Year_Birth']
    
    marketing_data_subset['Age_Group'] = pd.cut(
        marketing_data_subset['Age'],
        bins=[0, 25, 35, 45, 55, 65, 100],
        labels=['Gen Z', 'Early Millennial', 'Millennial', 'Gen X', 'Boomer', 'Senior']
    )
    
    print("Purchases Category Distribution:")
    display(marketing_data_subset['Purchases_Category'].value_counts().sort_index())
    
    print("\nAge Group Distribution:")
    display(marketing_data_subset['Age_Group'].value_counts().sort_index())
    
    # Visualize distributions
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    marketing_data_subset['Purchases_Category'].value_counts().sort_index().plot(kind='bar', ax=axes[0])
    axes[0].set_title('Purchase Categories')
    axes[0].set_xlabel('Category')
    axes[0].set_ylabel('Count')
    
    marketing_data_subset['Age_Group'].value_counts().sort_index().plot(kind='bar', ax=axes[1])
    axes[1].set_title('Age Groups')
    axes[1].set_xlabel('Age Group')
    axes[1].set_ylabel('Count')
    
    plt.tight_layout()
    plt.show()

### Technique 7: REMOVING DUPLICATE DATA

In [ ]:
if marketing_data_subset is not None:
    print("\n" + "="*70)
    print("DUPLICATE DETECTION & REMOVAL")
    print("="*70)
    
    # Check duplicates across all columns
    duplicate_check = marketing_data_subset.duplicated().sum()
    print(f"Total duplicate rows (full record): {duplicate_check}")
    
    # Check duplicates on key columns only
    id_dupes = marketing_data_subset['ID'].duplicated().sum()
    print(f"Duplicate IDs: {id_dupes}")
    
    # Remove duplicates
    cleaned_data = marketing_data_subset.drop_duplicates()
    print(f"\nOriginal shape: {marketing_data_subset.shape}")
    print(f"After deduplication: {cleaned_data.shape}")
    print(f"Rows removed: {marketing_data_subset.shape[0] - cleaned_data.shape[0]}")

### Technique 8: DROPPING ROWS & COLUMNS

In [ ]:
if cleaned_data is not None:
    print("\n" + "="*70)
    print("DROPPING: Removing Unwanted Data")
    print("="*70)
    
    # Drop specific columns
    cols_to_drop = ['Dt_Customer', 'Teenhome', 'Purchases_Category', 'Age', 'Age_Group']
    available_cols_to_drop = [col for col in cols_to_drop if col in cleaned_data.columns]
    
    data_after_drops = cleaned_data.drop(columns=available_cols_to_drop)
    print(f"Columns dropped: {available_cols_to_drop}")
    print(f"Shape after dropping: {data_after_drops.shape}")
    
    # Drop rows by condition
    data_filtered = data_after_drops[data_after_drops['Income'] > 0]
    print(f"\nFiltered data (Income > 0): {data_filtered.shape}")
    print(f"Rows with zero/negative income removed: {data_after_drops.shape[0] - data_filtered.shape[0]}")

### Technique 9: REPLACING DATA VALUES

In [ ]:
if data_filtered is not None:
    print("\n" + "="*70)
    print("REPLACING: Transforming Values")
    print("="*70)
    
    # Replace numeric teen indicator with text
    data_filtered['Has_Teen'] = data_filtered['Kidhome'].replace(
        {0: 'No Teen at Home', 1: 'Has Teen', 2: 'Multiple Teens'}
    )
    
    print("Value Replacement Examples:")
    display(data_filtered[['Kidhome', 'Has_Teen']].head(10))
    
    # Replace education levels
    if 'Education' in data_filtered.columns:
        education_map = {
            'Graduation': 'Bachelor/Diploma',
            'PhD': 'Doctorate',
            'Master': 'Master\'s Degree',
            'Basic': 'High School',
            '2n Cycle': 'Associate Degree'
        }
        data_filtered['Education_Clean'] = data_filtered['Education'].replace(education_map)
        print("\nEducation Mapping:")
        display(data_filtered[['Education', 'Education_Clean']].head())

### Technique 10: CHANGING DATA FORMAT

In [ ]:
if data_filtered is not None:
    print("\n" + "="*70)
    print("FORMAT CONVERSION: Data Type Changes")
    print("="*70)
    
    # Handle missing values before conversion
    if 'Income' in data_filtered.columns:
        data_filtered['Income_Filled'] = data_filtered['Income'].fillna(0)
        data_filtered['Income_Int'] = data_filtered['Income_Filled'].astype(int)
        
        print("Float to Int Conversion:")
        print(f"Original dtype: {data_filtered['Income'].dtype}")
        print(f"New dtype: {data_filtered['Income_Int'].dtype}")
        
        # Show comparison
        display(data_filtered[['Income', 'Income_Int']].head())
    
    # Convert year birth to age
    data_filtered['Current_Age'] = 2026 - data_filtered['Year_Birth']
    print(f"\nAge column added with dtype: {data_filtered['Current_Age'].dtype}")

### Technique 11: DEALING WITH MISSING VALUES

In [ ]:
if data_filtered is not None:
    print("\n" + "="*70)
    print("MISSING VALUE HANDLING")
    print("="*70)
    
    # Comprehensive missing value report
    missing_report = pd.DataFrame({
        'Column': data_filtered.columns,
        'Missing_Count': data_filtered.isnull().sum().values,
        'Missing_%': (data_filtered.isnull().sum().values / len(data_filtered) * 100).round(2)
    })
    
    display(missing_report[missing_report['Missing_Count'] > 0])
    
    # Drop missing values
    data_clean = data_filtered.dropna(how='any')
    print(f"\nAfter removing rows with any missing values:")
    print(f"Shape: {data_clean.shape}")
    print(f"Rows removed: {data_filtered.shape[0] - data_clean.shape[0]}")
    
    # Verify no missing values remain
    remaining_missing = data_clean.isnull().sum().sum()
    print(f"Remaining missing values: {remaining_missing}")

---
## PART 3: PREPARED DATA SUMMARY

In [ ]:
if data_clean is not None:
    print("\n" + "="*70)
    print("FINAL PREPARED DATASET SUMMARY")
    print("="*70)
    
    print(f"\nFinal Dataset Shape: {data_clean.shape}")
    print(f"Total Observations: {len(data_clean)}")
    print(f"Total Features: {len(data_clean.columns)}")
    
    print(f"\nColumns:")
    for i, col in enumerate(data_clean.columns, 1):
        print(f"  {i}. {col} ({data_clean[col].dtype})")
    
    print(f"\nStatistical Summary:")
    display(data_clean.describe(include='all').T)
    
    print("\n✅ Data preparation complete! Ready for exploratory analysis.")

---
## PART 4: SAVE PREPARED DATA

In [ ]:
if data_clean is not None:
    # Save prepared datasets
    OUTPUT_DIR = Path("prepared_data")
    OUTPUT_DIR.mkdir(exist_ok=True)
    
    # Save as CSV
    data_clean.to_csv(OUTPUT_DIR / "marketing_data_prepared.csv", index=False)
    print(f"✓ Prepared data saved to {OUTPUT_DIR}/marketing_data_prepared.csv")
    
    # Save as pickle for faster loading
    data_clean.to_pickle(OUTPUT_DIR / "marketing_data_prepared.pkl")
    print(f"✓ Prepared data saved as pickle to {OUTPUT_DIR}/marketing_data_prepared.pkl")
    
    print("\n💾 All preparation artifacts saved successfully!")